In [2]:
import pandas as pd
df = pd.read_csv('../Data/WELFake_Dataset.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Shape: (72134, 4)

Columns: ['Unnamed: 0', 'title', 'text', 'label']


In [3]:
print("Null values per column:\n", df.isnull().sum())
print("\nUnique Labels:", df['label'].value_counts())

Null values per column:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

Unique Labels: label
1    37106
0    35028
Name: count, dtype: int64


In [4]:
fake = df[df['label']==0][['title', 'text']].head(10)
for i, row in fake.iterrows():
    print(f"Fake Article{i}")
    print("Title:", row['title'])
    print("Text:", str(row["text"][:200]))
    print()

Fake Article3
Title: Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid
Text: A dozen politically active pastors came here for a private dinner Friday night to hear a conversion story unique in the context of presidential politics: how Louisiana Gov. Bobby Jindal traveled from 

Fake Article11
Title: May Brexit offer would hurt, cost EU citizens - EU parliament
Text: BRUSSELS (Reuters) - British Prime Minister Theresa May s offer of  settled status  for EU residents is flawed and will leave them with fewer rights after Brexit, the European Parliament s Brexit coor

Fake Article12
Title: Schumer calls on Trump to appoint official to oversee Puerto Rico relief
Text: WASHINGTON (Reuters) - Charles Schumer, the top Democrat in the U.S. Senate, called on President Donald Trump on Sunday to name a single official to oversee and coordinate relief efforts in hurricane-

Fake Article14
Title: No Change Expected for ESPN Political Agenda Des

In [5]:
real = df[df['label']==1][['title', 'text']].head(10)
for i, row in real.iterrows():
    print(f"Real Article{i}")
    print("Title", row['title'])
    print("Text:", str(row["text"][:200]))
    print()

Real Article0
Title LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]
Text: No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a 

Real Article1
Title nan
Text: Did they post their votes for Hillary already?

Real Article2
Title UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]
Text:  Now, most of the demonstrators gathered last night were exercising their constitutional and protected right to peaceful protest in order to raise issues and create change.    Loretta Lynch aka Eric H

Real Article4
Title SATAN 2: Russia unvelis an image of its terrifying new ‘SUPERNUKE’ – Western world takes notice
Text: The RS-28 Sarmat missile, dubbed Satan 2, will replace the SS-18 Flies at 4

In [6]:
print("Label distribution:\n", df['label'].unique())
print("Total articles:", len(df[df['label'] == 0]))
print("Total articles:", len(df[df['label'] == 1]))

Label distribution:
 [1 0]
Total articles: 35028
Total articles: 37106


In [7]:
import sys
print(sys.executable)

e:\TruthLensAI\venv\Scripts\python.exe


In [8]:
from lime.lime_text import LimeTextExplainer

In [9]:
import pickle

In [10]:
with open("../Model/pac_model.pkl", "rb") as f:
    pac_model = pickle.load(f)

with open("../Model/tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

e:\TruthLensAI\venv\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


In [11]:
def predict_proba(texts):
    vecs = tfidf.transform(texts)

    import numpy as np

    scores = pac_model.decision_function(vecs)
    probs = 1 / (1 + np.exp(-scores))
    return np.column_stack([1 - probs, probs])

explainer = LimeTextExplainer(class_names=['Real', 'Fake'])
exp = explainer.explain_instance("Some sample news text here", predict_proba, num_features=10)
print(exp.as_list())
exp = explainer.explain_instance("Some samplenews text here", predict_proba, num_features = 10)
print(exp.as_list())


[(np.str_('text'), -0.26179900590114175), (np.str_('news'), 0.15859588401688926), (np.str_('here'), -0.006125452590175468), (np.str_('Some'), -0.006045067666887531), (np.str_('sample'), -0.005774622644721879)]
[(np.str_('text'), -0.350051421920096), (np.str_('here'), 0.0002701593686181289), (np.str_('samplenews'), 0.0002449544303715297), (np.str_('Some'), 0.00023475163750210208)]


In [12]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from importlib import reload
import App.preprocessor

reload(App.preprocessor)

from App.preprocessor import clean_whatsapp

In [13]:
from App.preprocessor import clean_whatsapp

test_cases = [
    # Pure Hindi
    "यह खबर बिल्कुल सच है, कृपया सभी को भेजें 🙏",
    "सरकार ने नया कानून पास किया है!!!",

    # Mixed Hindi-English
    "Forwarded as received यह news सच है please share कर दो!!!",
    "BREAKING खबर: Modi govt ने FREE PETROL की घोषणा की!!!",

    # 50 exclamation marks
    "This is urgent" + "!" * 50,
    "SHARE THIS NOW" + "!" * 50,

    # Zero punctuation
    "this message has no punctuation at all just plain text",

    # Emoji-only
    "🙏🙏🙏😱😱🔥🔥🔥",
    "👍",

    # Very long (~2000 chars) — generate programmatically
    ("Forwarded as received " + "This is a very long WhatsApp forward. " * 50)[:2000],

    # ALL CAPS heavy
    "URGENT ALERT PLEASE READ THIS IMMEDIATELY AND SHARE",

    # Mixed case + numbers + links
    "Forwarded as received Check this http://fake-news-site.com NOW!!!",

    # Empty / whitespace only
    "",
    "     ",

    # Special characters / punctuation spam
    "***BREAKING***...!!!???...",

    # Single word
    "PETROL",

    # Repeated forwarded headers
    "Forwarded Forwarded as received this is fake news!!!",

    # Newlines and tabs mixed in
    "Forwarded as received\n\n\tBREAKING NEWS!!!\n\nShare now",

    # Mixed scripts + emoji + caps
    "अत्यावश्यक 🚨 URGENT खबर !!!! शेयर करें",

    # Non-latin punctuation (Hindi danda)
    "यह सच है। शेयर करें।",

    # Very short forward
    "fwd: true news",
]

for i, t in enumerate(test_cases, 1):
    result = clean_whatsapp(t)
    print(f"--- Test {i} ---")
    print(f"IN:  {t!r}")
    print(f"OUT: {result!r}")
    print()

--- Test 1 ---
IN:  'यह खबर बिल्कुल सच है, कृपया सभी को भेजें 🙏'
OUT: 'यह खबर बिल्कुल सच है, कृपया सभी को भेजें'

--- Test 2 ---
IN:  'सरकार ने नया कानून पास किया है!!!'
OUT: 'सरकार ने नया कानून पास किया है!'

--- Test 3 ---
IN:  'Forwarded as received यह news सच है please share कर दो!!!'
OUT: 'Forwarded as received यह news सच है please share कर दो!'

--- Test 4 ---
IN:  'BREAKING खबर: Modi govt ने FREE PETROL की घोषणा की!!!'
OUT: 'Breaking खबर: Modi govt ने Free Petrol की घोषणा की!'

--- Test 5 ---
IN:  'This is urgent!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'
OUT: 'This is urgent!'

--- Test 6 ---
IN:  'SHARE THIS NOW!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'
OUT: 'Share This Now!'

--- Test 7 ---
IN:  'this message has no punctuation at all just plain text'
OUT: 'this message has no punctuation at all just plain text'

--- Test 8 ---
IN:  '🙏🙏🙏😱😱🔥🔥🔥'
OUT: ''

--- Test 9 ---
IN:  '👍'
OUT: ''

--- Test 10 ---
IN:  'Forwarded as received This is a very long WhatsApp fo

In [ ]:
import sys, os
sys.path.append(os.path.abspath('...'))
from App.preprocessor import detect_language

print(detect_language("Modi announces free petrol for tye citezen"))
print(detect_language("यह खबर बिल्कुल सच है, कृपया सभी को भेजें"))
print(detect_language("Ye news bilkul sach hai, please share kar do"))

en
hi
en


In [17]:
import sys, os
sys.path.append(os.path.abspath('..'))
from App.translator import translate_to_english

hindi_sentences = [
    "यह खबर बिल्कुल सच है, कृपया सभी को भेजें",
    "सरकार ने नया कानून पास किया है",
    "मोदी ने मुफ्त पेट्रोल की घोषणा की",
    "यह जानकारी झूठी है, कृपया सत्यापित करें",
    "आज मौसम बहुत अच्छा है"
]

for s in hindi_sentences:
    translated = translate_to_english
    print(f"Breaking News:{s}")
    print(f"EN:{translated}")
    print()

Breaking News:यह खबर बिल्कुल सच है, कृपया सभी को भेजें
EN:<function translate_to_english at 0x0000025924450900>

Breaking News:सरकार ने नया कानून पास किया है
EN:<function translate_to_english at 0x0000025924450900>

Breaking News:मोदी ने मुफ्त पेट्रोल की घोषणा की
EN:<function translate_to_english at 0x0000025924450900>

Breaking News:यह जानकारी झूठी है, कृपया सत्यापित करें
EN:<function translate_to_english at 0x0000025924450900>

Breaking News:आज मौसम बहुत अच्छा है
EN:<function translate_to_english at 0x0000025924450900>

